[Guide Index](README.md) | [Appendix A](appendix-a.ipynb)


# Appendix A2: Named, Shared Parts in `sqe:` Syntax Trees

The [SPARQL queries as RDF guide](05b-sparql-query-as-rdf.ipynb) describes how StarLayer treasts SPARQL queries as syntax tree graphs using the `sqe:` namespace.  

As graph thse `sqe` syntax trees can be manipulated as graphs.  The following example shows how two queries can share named component parts. 

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.

In [ ]:
from starlayer.graph import StarLayerGraph, Namespace
from starlayer.sparql import sqe
from rdflib import URIRef

EX = Namespace("http://example.org/")


## 1. Two queries, parsed separately - each with its own filter.

A small data graph is created for product shipments.

Then two similar queries are created, one for shipped date and one for received date.  These are covverted into `sqe` syntact tree graphs.  


In [1]:
from starlayer.graph import StarLayerGraph, Namespace
from starlayer.sparql import sqe
from rdflib import URIRef

EX = Namespace("http://example.org/")

shipments = StarLayerGraph()
shipments.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:ship1 ex:shipDate "2026-01-05"^^xsd:date ; ex:receivedDate "2026-01-12"^^xsd:date .
    ex:ship2 ex:shipDate "2026-02-10"^^xsd:date ; ex:receivedDate "2026-02-15"^^xsd:date .
    ex:ship3 ex:shipDate "2026-03-01"^^xsd:date ; ex:receivedDate "2026-03-04"^^xsd:date .
""", format="turtle")
shipments.bind("ex", EX)

In [2]:
root_shipped = URIRef("http://example.org/queries#shippedSinceCutoff")
root_received = URIRef("http://example.org/queries#receivedSinceCutoff")

q_shipped_text = """PREFIX ex: <http://example.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?s WHERE { ?s ex:shipDate ?d . FILTER(?d >= "2026-02-01"^^xsd:date) }"""

q_received_text = """PREFIX ex: <http://example.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?s WHERE { ?s ex:receivedDate ?d . FILTER(?d >= "2026-02-01"^^xsd:date) }"""

private_graph, _ = sqe.sqe_parse_to_tree(q_shipped_text, root=root_shipped)
private_graph, _ = sqe.sqe_parse_to_tree(q_received_text, graph=private_graph, root=root_received)

private_graph.bind("sqe", sqe.SQE)
private_graph.bind("q", "http://example.org/queries#")
print(private_graph.serialize(format="turtle12"))

@prefix ns1: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix ns2: <http://example.org/> .
@prefix q: <http://example.org/queries#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

q:receivedSinceCutoff a sqe:SelectQuery, sqe:Query ;
    sqe:projection ( "s"^^ns1:Variable ) ;
    sqe:prologuePrefix [
        a sqe:PrefixBinding ;
        sqe:namespace xsd: ;
        sqe:prefixLabel "xsd"^^ns1:PlainString
    ], [
        a sqe:PrefixBinding ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^ns1:PlainString
    ] ;
    sqe:where ( [
        a sqe:TriplePattern ;
        sqe:object "d"^^ns1:Variable ;
        sqe:predicate ns2:receivedDate ;
        sqe:subject "s"^^ns1:Variable
    ] [
        a sqe:Filter ;
        sqe:expr [
            a ns1:RelationalExpression ;
            ns1:expr "d"^^ns1:Variable ;
            ns1:op ">="^^ns1:PlainString ;
     

## 2. Loading the graph directly - two queries sharing one named filter

Those same two queries can be represented as follows.  See that the identical filters have been replaced by a named entity.  

[q:sinceCutoffFilter a sqe:Filter ;
    sqe:expr [
        a spexpr:RelationalExpression ;
        spexpr:expr "d"^^spexpr:Variable ;
        spexpr:op ">="^^spexpr:PlainString ;
        spexpr:other "2026-02-01"^^xsd:date
    ] .

In [3]:
turtle_before = """
@prefix ex: <http://example.org/> .
@prefix q: <http://example.org/queries#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

q:shippedSinceCutoff a sqe:Query, sqe:SelectQuery ;
    sqe:projection ( "s"^^spexpr:Variable ) ;
    sqe:prologuePrefix [ a sqe:PrefixBinding ; sqe:namespace ex: ; sqe:prefixLabel "ex"^^spexpr:PlainString ] ;
    sqe:where ( [
        a sqe:TriplePattern ;
        sqe:subject "s"^^spexpr:Variable ;
        sqe:predicate ex:shipDate ;
        sqe:object "d"^^spexpr:Variable
    ] q:sinceCutoffFilter ) .

q:receivedSinceCutoff a sqe:Query, sqe:SelectQuery ;
    sqe:projection ( "s"^^spexpr:Variable ) ;
    sqe:prologuePrefix [ a sqe:PrefixBinding ; sqe:namespace ex: ; sqe:prefixLabel "ex"^^spexpr:PlainString ] ;
    sqe:where ( [
        a sqe:TriplePattern ;
        sqe:subject "s"^^spexpr:Variable ;
        sqe:predicate ex:receivedDate ;
        sqe:object "d"^^spexpr:Variable
    ] q:sinceCutoffFilter ) .

q:sinceCutoffFilter a sqe:Filter ;
    sqe:expr [
        a spexpr:RelationalExpression ;
        spexpr:expr "d"^^spexpr:Variable ;
        spexpr:op ">="^^spexpr:PlainString ;
        spexpr:other "2026-02-01"^^xsd:date
    ] .
"""

shared_graph = StarLayerGraph()
shared_graph.parse(data=turtle_before, format="turtle12")
print("triples parsed:", len(shared_graph))

conforms, _, report_text = sqe.sqe_validate(shared_graph)
print("conforms:", conforms)

triples parsed: 42


conforms: True


## 3. Execute both queries

Both share the same 2026-02-01 cutoff, just against a different predicate:

In [4]:
q_shipped_obj = sqe.sqe_tree_to_query(shared_graph, root_shipped)
q_received_obj = sqe.sqe_tree_to_query(shared_graph, root_received)

print("shipped since cutoff:", [shipments.qname(r.s) for r in shipments.query(q_shipped_obj)])
print("received since cutoff:", [shipments.qname(r.s) for r in shipments.query(q_received_obj)])

shipped since cutoff: ['ex:ship2', 'ex:ship3']
received since cutoff: ['ex:ship2', 'ex:ship3']


## 4. Edit the shared filter - find it, delete it (CBD), replace it

Because `q:sinceCutoffFilter` is a named entity, it can be located, and replaced.  This example uses `.cbd` to return all triples that are descendents of `q:sinceCutoffFilter`.


In [30]:
filter_node = URIRef("http://example.org/queries#sinceCutoffFilter")

# delete the node and it descendents
cbd = shared_graph.cbd(filter_node)
print("CBD triples to remove:", len(cbd))
shared_graph -= cbd

# add a new filter.
new_filter = """
@prefix q: <http://example.org/queries#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

q:sinceCutoffFilter a sqe:Filter ;
    sqe:expr [
        a spexpr:RelationalExpression ;
        spexpr:expr "d"^^spexpr:Variable ;
        spexpr:op ">="^^spexpr:PlainString ;
        spexpr:other "2026-03-01"^^xsd:date
    ] .
"""
# add the new filter.
shared_graph.parse(data=new_filter, format="turtle12")

#check for validitity
conforms, _, report_text = sqe.sqe_validate(shared_graph)
print("conforms:", conforms)

CBD triples to remove: 6
conforms: True


Output and execute just the *received* query:

In [6]:
# output: render the received-query back to text
print(sqe.sqe_tree_to_text(shared_graph, root_received))

# execute: decode and run that same query
q_received_edited = sqe.sqe_tree_to_query(shared_graph, root_received)
print("received since cutoff, after edit:", [shipments.qname(r.s) for r in shipments.query(q_received_edited)])

PREFIX ex: <http://example.org/>
SELECT ?s
WHERE { ?s ex:receivedDate ?d . FILTER(?d >= "2026-03-01"^^<http://www.w3.org/2001/XMLSchema#date>) }
received since cutoff, after edit: ['ex:ship3']
